# Introduction to Large Language Models (LLMs)


## What is a Large Language Model?

Large Language Models (LLMs) are deep learning models trained on massive text corpora. They can understand and generate human-like text.

Popular LLMs include:

- GPT (OpenAI)
- Gemini (Google)
- Claude (Anthropic)
- LLaMA (Meta)
- DeepSeek

## Applications of LLMs

LLMs can be used in many tasks:

- Text summarization
- Translation
- Question answering
- Text generation
- Chatbots
- Sentiment analysis
- Code generation


## Limitations and Ethical Considerations

- May generate factually incorrect answers ("hallucination")
- Can reflect biases in training data
- Large environmental and computational cost
- Needs context-appropriate prompting

## Open-Source vs Commercial LLMs: Access and Use

LLMs come in two main forms:

### Commercial (Proprietary) APIs
- Closed weights
- Usually accessed via API
- Paid (may include free tier)
- Strong performance, frequent updates
- Examples: OpenAI GPT-4, Anthropic Claude, Google Gemini

###  Open-Source Models
- Weights are available for download
- Can be run locally or on your own infrastructure
- Community-supported, customizable
- Examples: Meta's LLaMA, Mistral, Falcon, DeepSeek


## Multimodal LLMs

Transformers can be used to process [text](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) as well as [images](https://arxiv.org/abs/2010.11929) or other types of [data](https://arxiv.org/abs/2205.06175). 

Recently, LLMs are becoming multimodal and can process text and images in an integrated fashion.

LLMs are based on a neural network architecture called a **Transformer**, introduced in 2017 ([Attention is all you need, Vaswani et al. 2017](https://arxiv.org/pdf/1706.03762.pdf))

## Attention and transformers

In a self-attention layer, an input matrix $X$ ($n$ tokens of dimension $d$) are turned it into an output matrix $Z$ ($n$ components of dimension $d_v$) via three representational matrices of the input:

* queries Q
* keys K
* values V

$\Large {\rm Attention}(Q, K, V) = {\rm softmax}( Q \cdot K^T / \sqrt{d_k}) * V$

where $Q$, $K$ and $V$ are matrices representing linear transformations from the input matrix $x$ via learnable parameters $W^Q$, $W^K$ and $W^V$:

* $Q = X W^Q$
* $K = X W^K$
* $V = X W^V$

Note that 
* $X \in \mathbb{R}^{n \times d}$
* $Q \in \mathbb{R}^{n \times d_k}$
* $K \in \mathbb{R}^{n \times d_k}$
* $V \in \mathbb{R}^{n \times d_v}$
* $W^Q \in \mathbb{R}^{d \times d_k}$
* $W^K \in \mathbb{R}^{d \times d_k}$
* $W^V \in \mathbb{R}^{d_v \times d}$
* $Z \in \mathbb{R}^{n \times d_v}


**Self-attention:**

![self-attention](selfattention.png)


**Cross-attention:**

In cross-attention, an input matrix $X_1$ ($n$ tokens of dimension $d$) is turned it into an output matrix $Z$ ($n$ components of dimension $d_v$) contrasting with another input matrix $X_2$ via three representational matrices of the input, where we now have:

* $Q = X_1 W^Q$
* $K = X_2 W^K$
* $V = X_2 W^V$

Note that 
* $X_1 \in \mathbb{R}^{n \times d}$
* $X_2 \in \mathbb{R}^{m \times d}$
* $Q \in \mathbb{R}^{n \times d_k}$
* $K \in \mathbb{R}^{m \times d_k}$
* $V \in \mathbb{R}^{m \times d_v}$
* $W^Q \in \mathbb{R}^{d \times d_k}$
* $W^K \in \mathbb{R}^{d \times d_k}$
* $W^V \in \mathbb{R}^{d_v \times d}$
* $Z \in \mathbb{R}^{n \times d_v}

![cross-attention](cross-attention-summary.png)

A transformer uses several multi-head-attention layers to perform tasks such as translation, next token prediction, or even image classification.

![transformer](transformer.png)

In the case of the next token prediction, e.g. GPT, we only use the decoder part. 

During training, tokens are shifted one element to the right to be compared to the original values (the values to predict), properly masked to prevent the decoder from seeing future tokens. 

During inference, we predict one token at a time.

Example:

* Input tokens (shifted right):

```[CLS] The dog chased```

* Target tokens (what to predict):

```The dog chased the```

So the model learns:

From [CLS] → predict "The"

From "The" → predict "dog"

From "dog" → predict "chased"

From "chased" → predict "the"



### Masked self-atention

The attention formula is modified as follows:

$\Large {\rm Attention}(Q, K, V) = {\rm softmax}( Q \cdot K^T / \sqrt{d_k} + M) * V$

with M being a mask matrix, e.g.

|         | [CLS] | The | dog | chased |
| ------- | --- | --- | --- | --- |
| [CLS] | 0   | −∞  | −∞  | −∞  |
| The | 0   | 0   | −∞  | −∞  |
| dog | 0   | 0   | 0   | −∞  |
| chased | 0   | 0   | 0   | 0   |

so the softmax returns 0 for all future tokens in each row (they cannot pay attention to the future).


## Training LLMs

Large Language Models are usually trained in three phases:

### Unsupervised Pre-Training

We maximize the likelihood:

$\Large \sum_i \log P(u_i | u_{i-k}, ..., u_{i-1}; \theta)$

where we use a corpus of tokens $U=\lbrace{u_1, ..., u_n\rbrace}$

and where a Transformer Decoder Memory Compressed Attention ([T-DMCA](https://arxiv.org/abs/1801.10198)) model is used. It modified the transformer in three ways:

1. Decoder only: the encoder layer is removed to do next token prediction
2. Memory-compressed attention: the number of keys and values are reduced by doing a strided convolution.
3. Local attention: it divides the tokens into blocks of similar length and attention is performed in each block independently

![TDMCA](TDMCA.png)

### Supervised training


![openai](GPT.png)


The pretrained model can be modified and fine-tuned to solve some supervised tasks such as classification (sentiment analysis), entailment (logic analysis), similarity, and multiple choice in a supervised fashion ([Radford et al. 2018 (GPT)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)).



### Reinforcement Learning with Human Feedback (RLHF)

🔹 Supervised Fine-Tuning (SFT)

Human labelers provide example prompts and ideal responses.

The model is fine-tuned on this data to become better aligned with user expectations.

🔹 Reward Model (RM)

Collect human comparisons between two or more model outputs for the same prompt.

Train a separate reward model to predict which output is preferred.

E.g., "Output A is better than B" → RM learns a scoring function.

🔹 Reinforcement Learning (PPO)

Use Proximal Policy Optimization (PPO) (a reinforcement learning algorithm) to optimize the LLM so that its outputs maximize the reward model score.

The LLM becomes a policy that chooses tokens, and the RM guides it toward preferred behavior.

### Full pipeline

The full pipeline looks like this:

* Pretraining

Train a transformer on a massive dataset using next-token prediction (e.g., GPT-style language modeling).

* Supervised Fine-Tuning (SFT)

Fine-tune the model using human-written demonstrations of ideal behavior (e.g., answering politely, correcting mistakes).

* RLHF (Reinforcement Learning from Human Feedback)

Use reinforcement learning to further refine the model using human preference judgments.

##  Summarization with Hugging Face Transformers

We will use the `transformers` library to run a pre-trained summarization model: `facebook/bart-large-cnn`.


In [1]:
!pip install transformers -q

In [2]:
from transformers import pipeline

# Load a summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

/home/fforster/anaconda3/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/fforster/anaconda3/lib/python3.8/site-packages/transformers/modeling_utils.py:415: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by th

In [3]:
article = """
ALeRCE is a real-time astronomical alert broker designed to process data from the Vera C. Rubin Observatory.
It classifies variable and transient phenomena such as supernovae and variable stars using machine learning algorithms.
It provides web interfaces and APIs to facilitate scientific exploration.
"""

summary = summarizer(article, max_length=50, min_length=10, do_sample=False)
print(summary[0]['summary_text'])

ALeRCE is a real-time astronomical alert broker. It classifies variable and transient phenomena such as supernovae and variable stars.


In [4]:
# Install OpenAI SDK
!pip install openai -q

In [5]:
from openai import OpenAI

client = OpenAI(api_key=open("openai.key").read().strip())  # your 164-char key

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Hello, are you working?"}
    ]
)

print(response.choices[0].message.content)

Hello! I am always here to help you. How can I assist you today?


In [6]:
response

ChatCompletion(id='chatcmpl-CQZlzckA3KqFCZGzBmtZzPi40rz3r', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! I am always here to help you. How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1760450075, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=17, prompt_tokens=13, total_tokens=30, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [7]:
!pip install anthropic -q

In [9]:
import anthropic

client = anthropic.Anthropic(api_key=open("claude.key").read().strip("\n"))

response = client.messages.create(
    model="claude-3-opus-20240229",
    max_tokens=500,
    temperature=0.5,
    messages=[
        {"role": "user", "content": "Explain the purpose of the ALeRCE project in simple terms."}
    ]
)

print(response.content[0].text)


The ALeRCE (Automatic Learning for the Rapid Classification of Events) project is an initiative that aims to quickly detect and classify transient astronomical events, such as supernovae, using machine learning techniques. Its main purpose is to efficiently process and analyze the vast amounts of data generated by modern astronomical surveys, enabling astronomers to identify and study these events in near real-time.

In simple terms, ALeRCE acts as a cosmic "alert system" that scans the night sky for sudden changes or new objects appearing. When a potential transient event is detected, the system automatically classifies it based on its characteristics and sends out notifications to astronomers worldwide. This allows researchers to quickly follow up on interesting events and gather more data before the objects fade away.

By automating the detection and classification process, ALeRCE helps astronomers manage the overwhelming amount of data produced by modern telescopes and focus their 

In [10]:
response

Message(id='msg_015SWg5XZSGyYttBL9FKsGco', content=[TextBlock(citations=None, text='The ALeRCE (Automatic Learning for the Rapid Classification of Events) project is an initiative that aims to quickly detect and classify transient astronomical events, such as supernovae, using machine learning techniques. Its main purpose is to efficiently process and analyze the vast amounts of data generated by modern astronomical surveys, enabling astronomers to identify and study these events in near real-time.\n\nIn simple terms, ALeRCE acts as a cosmic "alert system" that scans the night sky for sudden changes or new objects appearing. When a potential transient event is detected, the system automatically classifies it based on its characteristics and sends out notifications to astronomers worldwide. This allows researchers to quickly follow up on interesting events and gather more data before the objects fade away.\n\nBy automating the detection and classification process, ALeRCE helps astronome

In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.deepseek.com/v1",
    api_key=open("deepseek.key").read().strip()
)

response = client.chat.completions.create(
    model="deepseek-chat",  # or "deepseek-coder"
    messages=[
        {"role": "system", "content": "You are an expert assistant in astronomy."},
        {"role": "user", "content": "What is a Type Ia supernova?"}
    ]
)

print(response.choices[0].message.content)


Of course! Here is a detailed explanation of a Type Ia supernova.

### The Short Answer

A **Type Ia supernova** (pronounced "type one-A") is a cataclysmic explosion of a white dwarf star. It is one of the most important and energetic events in the universe, serving as a crucial **"standard candle"** for measuring cosmic distances.

---

### The Detailed Explanation

#### 1. The Progenitor System: How It Forms

A Type Ia supernova doesn't come from a single, massive star. Instead, it occurs in a **binary star system** (two stars orbiting each other). The classic scenario involves:

*   **A White Dwarf:** This is the super-dense, Earth-sized core of a star like our Sun that has run out of fuel and died. It's primarily made of carbon and oxygen and is supported against gravity by a quantum mechanical effect called **electron degeneracy pressure**.
*   **A Companion Star:** This can be a regular star (like our Sun) or another white dwarf.

The white dwarf's intense gravity pulls material,

In [12]:
response

ChatCompletion(id='798348f3-c2d4-4f1e-9f25-e0fdf6d6cc1b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Of course! Here is a detailed explanation of a Type Ia supernova.\n\n### The Short Answer\n\nA **Type Ia supernova** (pronounced "type one-A") is a cataclysmic explosion of a white dwarf star. It is one of the most important and energetic events in the universe, serving as a crucial **"standard candle"** for measuring cosmic distances.\n\n---\n\n### The Detailed Explanation\n\n#### 1. The Progenitor System: How It Forms\n\nA Type Ia supernova doesn\'t come from a single, massive star. Instead, it occurs in a **binary star system** (two stars orbiting each other). The classic scenario involves:\n\n*   **A White Dwarf:** This is the super-dense, Earth-sized core of a star like our Sun that has run out of fuel and died. It\'s primarily made of carbon and oxygen and is supported against gravity by a quantum mechanical effect called *

## Agentic AI: Language Models That Act

LLMs can also act as **agents**: they reason, plan, and take actions (e.g., query tools, search, code, run tasks) in a loop.

This approach powers tools like:
- AutoGPT
- LangChain agents
- CrewAI
- OpenAI Functions / Tools
- PydanticAI

Let's try `langchain` and an OpenAI-compatible API (e.g., GPT-4, DeepSeek).


In `langchain` one can specify the reasoning strategies:

| Agent Type               | Description                                          |
| ------------------------ | ---------------------------------------------------- |
| `zero-shot-react`        | ReAct-style prompt, chooses tools via text reasoning |
| `chat-zero-shot-react`   | Same, but using chat models like GPT-4               |
| `openai-functions-agent` | Uses OpenAI's function calling                       |
| `structured-chat`        | Uses structured outputs for tool selection           |
| `plan-and-execute`       | Makes a plan, then executes step-by-step             |
| `self-ask-with-search`   | First asks clarifying questions, then answers        |


In `langchain` one needs to specify which tools are available to use:

| Tool Name               | Purpose                                    |
| ----------------------- | ------------------------------------------ |
| `llm-math`              | Performs math using a language model       |
| `serpapi` or `requests` | Web search                                 |
| `python`                | Executes Python code safely                |
| `openai-functions`      | Interacts with OpenAI Function Calling     |
| `wikipedia`             | Queries Wikipedia                          |
| `vectorstore`           | Searches a vector DB (e.g., for RAG)       |
| `terminal` (dangerous!) | Runs shell commands                        |
| `sql`                   | Queries a database                         |
| `toolkits`              | Bundles like for Pandas, SQL, OpenAI, etc. |


In [13]:
!pip install langchain -q

In [14]:
!pip install wikipedia

In [15]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import load_tools, initialize_agent, get_all_tool_names
from langchain.agents.agent_types import AgentType
import os

In [16]:
# Use your DeepSeek API credentials
os.environ["OPENAI_API_KEY"] = open("deepseek.key").read().strip()
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com"

# LLM with DeepSeek via OpenAI-compatible wrapper
llm = ChatOpenAI(model="deepseek-chat")

/home/fforster/anaconda3/lib/python3.8/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [17]:
# Show all tools that can be loaded by name
tool_names = get_all_tool_names()
print(sorted(tool_names))

sel_tools = []
for tool in tool_names:
    try:
        load_tools([tool], llm=llm)
        sel_tools.append(tool)
    except:
        print(f"Warning {tool}")

tools = load_tools(sel_tools, llm=llm)

# Print the description of each tool
for name, tool in zip(sel_tools, tools):
    print(name)
    try:
        print(f"Tool name: {tool.name}")
        print(f"Description: {tool.description}")
        print("-" * 60)
    except:
        print(tool)

['arxiv', 'awslambda', 'bing-search', 'dalle-image-generator', 'dataforseo-api-search', 'dataforseo-api-search-json', 'ddg-search', 'eleven_labs_text2speech', 'golden-query', 'google-finance', 'google-jobs', 'google-lens', 'google-scholar', 'google-search', 'google-search-results-json', 'google-serper', 'google-serper-results-json', 'google-trends', 'google_cloud_texttospeech', 'graphql', 'human', 'llm-math', 'memorize', 'merriam-webster', 'metaphor-search', 'news-api', 'open-meteo-api', 'openweathermap-api', 'podcast-api', 'pubmed', 'read_file', 'reddit_search', 'requests', 'requests_delete', 'requests_get', 'requests_patch', 'requests_post', 'requests_put', 'sceneXplain', 'searchapi', 'searchapi-results-json', 'searx-search', 'searx-search-results-json', 'serpapi', 'sleep', 'stackexchange', 'terminal', 'tmdb-api', 'twilio', 'wikipedia', 'wolfram-alpha']
Warning wolfram-alpha
Warning google-search
Warning google-search-results-json
Warning searx-search-results-json
Warning bing-search

In [18]:
# Load simple tools (calculator, Wikipedia, etc.)
tools = load_tools(["wikipedia", "llm-math"], llm=llm)

# Create an agent
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

/home/fforster/anaconda3/lib/python3.8/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [19]:
# Ask the agent a reasoning task
response = agent.run("What is the Hubble constant and how does it relate mathematically to redshift?")

/home/fforster/anaconda3/lib/python3.8/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
Thought: I need to understand what the Hubble constant is and its mathematical relationship to redshift. I should start by looking up the Hubble constant on Wikipedia.

Action: wikipedia  
Action Input: Hubble constant  

Observation: Page: Hubble's law
Summary: Hubble's law, also known as the Hubble–Lemaître law, is the observation in physical cosmology that galaxies are moving away from Earth at speeds proportional to their distance. In other words, the farther a galaxy is from the Earth, the faster it moves away. A galaxy's recessional velocity is typically determined by measuring its redshift, a shift in the frequency of light emitted by the galaxy.
The discovery of Hubble's law is attributed to work published by Edwin Hubble in 1929, but the notion of the universe expanding at a calculable rate was first derived from general relativity equations in 1922 by Alexander Friedmann. The Friedmann equations showed the universe might be expanding, a

Thought:Thought: The Wikipedia entries provide a good overview, but I need to clarify the mathematical relationship between redshift and the Hubble constant. Redshift (z) is related to recessional velocity (v), and Hubble's law states v = H₀ * D. For small redshifts, the recessional velocity is approximately v ≈ c * z, where c is the speed of light. Combining these, we get H₀ * D ≈ c * z. For larger redshifts, the relationship becomes more complex due to relativistic effects. I should confirm this with a more specific search.

Action: wikipedia  
Action Input: redshift recessional velocity Hubble law  

Observation: Page: Hubble's law
Summary: Hubble's law, also known as the Hubble–Lemaître law, is the observation in physical cosmology that galaxies are moving away from Earth at speeds proportional to their distance. In other words, the farther a galaxy is from the Earth, the faster it moves away. A galaxy's recessional velocity is typically determined by measuring its redshift, a shif

In [20]:
response

"The Hubble constant (H₀) is the constant of proportionality in Hubble's law, which states that the recessional velocity (v) of a galaxy is proportional to its distance (D): v = H₀ * D. It is typically measured in km/s/Mpc, and its current value is around 70 km/s/Mpc. Redshift (z) is a measurable quantity related to the recessional velocity. For small redshifts (z << 1), the relationship is approximately v ≈ c * z, where c is the speed of light. Combining this with Hubble's law gives H₀ * D ≈ c * z, or z ≈ (H₀ / c) * D. Thus, redshift is directly proportional to distance via the Hubble constant. For larger redshifts, relativistic corrections are needed, and the exact relationship involves the Hubble parameter over time, but the simple linear approximation is often used for nearby galaxies."

In [21]:
response = agent.run("Who was Albert Einstein's wife and how many years ago was she born? It is very important to confirm what is the current year first.")



> Entering new AgentExecutor chain...
Thought: I need to find out who Albert Einstein's wife was and her birth year, then calculate how many years ago she was born. But first, I need to confirm the current year to make the calculation accurate. Since I don't have a direct way to get the current year, I'll assume it's 2024 for now, as that's a common reference point. However, I should use the Wikipedia tool to get information about Albert Einstein's wife and her birth date.

Action: wikipedia  
Action Input: Albert Einstein  

Observation: Page: Albert Einstein
Summary: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation". He received the 1921 Nobel Prize in Physics for "his services to theoretical physics,

In [22]:
response

"** Albert Einstein's wife was Mileva Marić, and she was born 149 years ago (as of 2024)."

In [ ]:
response = agent.run("I want to understand the impact of rain on car accidents. Can you propose what factors can impact the number of car accidents and their causal relation?")



> Entering new AgentExecutor chain...
Thought: The user is asking about the impact of rain on car accidents and wants to understand the causal factors involved. This is a complex topic involving road safety, weather, and human behavior. I should start by gathering general information about how rain affects driving conditions and accident rates. The Wikipedia tool would be a good starting point to get an overview of factors like road slipperiness, visibility, and driver behavior in rain.

Action: wikipedia
Action Input: "effect of rain on car accidents road safety"
Observation: Page: Safety car
Summary: In motorsport, a safety car, or a pace car, is a car that limits the speed of competing cars or motorcycles on a racetrack in the case of a caution period, such as an obstruction on the track or bad weather. The safety car aims to enable the clearance of any obstruction under safer conditions, especially for marshals and/or awaiting more favourable track conditions weather-wise. By fol

In [28]:
response

"**  \n\nRain increases car accidents through several mechanisms:  \n\n1. **Reduced Traction** → Longer braking distances, higher skid risk.  \n2. **Poor Visibility** → Delayed reaction times, lane departures.  \n3. **Driver Errors** → Failure to adapt speed/distance to conditions.  \n4. **Hydroplaning** → Loss of control at high speeds.  \n5. **Road Hazards** → Flooding, oil slicks, or obscured markings.  \n\nCausal chains include:  \n- Rain → Slippery roads → More rear-end collisions.  \n- Rain + Speeding → Hydroplaning → Off-road crashes.  \n- Rain + Nighttime → Low visibility → Pedestrian accidents.  \n\nFor exact statistics or braking distance calculations, I can use tools like **Wikipedia** or **Calculator**. Let me know if you'd like specific data!"

## How LangChain Agents Decide What to Do

LangChain uses LLMs as decision-makers. The process typically follows a reasoning pattern like:

User question → LLM interprets it → Calls a tool if needed → Responds

* The Agent is a Prompted LLM

LangChain wraps an LLM (like GPT-4 or DeepSeek) with a special prompt template that encourages:

Step-by-step reasoning (like in ReAct: Reasoning + Acting)

Decision-making (e.g., should I search, calculate, or just respond?)

* Available Tools are Described in the Prompt

You provide LangChain a list of tools (functions like search(), calculator(), Python REPL, etc.), and it tells the LLM:

"Here’s a question."

"Here are tools you can use."

"Decide whether to use one, or just answer."

* The Agent’s Workflow (ReAct Loop)

For AgentType.ZERO_SHOT_REACT_DESCRIPTION, the agent works like this:

Question: What is the distance to Andromeda?

Thought: I should look this up online.
Action: Wikipedia
Action Input: Andromeda galaxy distance

Observation: 2.5 million light-years

Thought: I now know the answer.
Final Answer: The distance to Andromeda is approximately 2.5 million light-years.

*  What is llm-math?

It's part of LangChain's built-in tools and works like this:

The LLM reads the user's question (e.g., “What is 23.5% of 894?”).

It decides it can’t calculate this reliably in text.

It calls the llm-math tool, which:

Uses the LLM to generate a Python expression (0.235 * 894)

Executes the code in a safe Python environment

Returns the result to the agent

So it's like giving the LLM a calculator powered by Python.